# NCAA Baseline Reproducibility Notebook

This notebook runs the modular Phase 0-6 pipeline via `src/train_baseline.py`.

It now includes:
- base models + rolling OOF
- stacking (`stack`, `stack_tourney`)
- rolling calibration sweep (`platt`, `isotonic`, `temperature`) and best calibrated OOF exports
- Phase 6 graph embeddings/features with optional cached embedding reuse (`NCAA_PHASE6_RETRAIN=0` by default).

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'train_baseline.py').exists() and (ROOT.parent / 'src' / 'train_baseline.py').exists():
    ROOT = ROOT.parent

print('Workspace:', ROOT)
print('Train script exists:', (ROOT / 'src' / 'train_baseline.py').exists())

In [ ]:
import os
import subprocess
import sys

from pathlib import Path

root = Path.cwd().resolve()
if not (root / 'src' / 'train_baseline.py').exists() and (root.parent / 'src' / 'train_baseline.py').exists():
    root = root.parent

env = os.environ.copy()
env.setdefault('NCAA_ENABLE_EXTRA_MODELS', '1')
env.setdefault('NCAA_STACK_TOURNEY_WEIGHT', '1.0')
env.setdefault('NCAA_CAL_METHODS', 'platt,isotonic,temperature')
env.setdefault('NCAA_CAL_SCOPES', 'all,tournament_only')
env.setdefault('NCAA_CAL_SHRINKS', '0.0,0.08')

# Phase 6 toggles
env.setdefault('NCAA_PHASE6_ENABLE', '1')
env.setdefault('NCAA_PHASE6_RETRAIN', '0')
env.setdefault('NCAA_PHASE6_EMBEDDING_DIM', '64')
env.setdefault('NCAA_PHASE6_GNN_LAYERS', '2')
env.setdefault('NCAA_PHASE6_EPOCHS', '30')
env.setdefault('NCAA_PHASE6_LR', '0.001')
env.setdefault('NCAA_PHASE6_INCLUDE_TOURNEY_EDGES', '0')
env.setdefault('NCAA_PHASE6_USE_GRAPH_WINPROB', '0')

cmd = [sys.executable, str(root / 'src' / 'train_baseline.py')]
print('Running:', ' '.join(cmd))
print('Using env overrides:')
for k in [
    'NCAA_ENABLE_EXTRA_MODELS',
    'NCAA_STACK_TOURNEY_WEIGHT',
    'NCAA_CAL_METHODS',
    'NCAA_CAL_SCOPES',
    'NCAA_CAL_SHRINKS',
    'NCAA_PHASE6_ENABLE',
    'NCAA_PHASE6_RETRAIN',
    'NCAA_PHASE6_EMBEDDING_DIM',
    'NCAA_PHASE6_GNN_LAYERS',
    'NCAA_PHASE6_EPOCHS',
    'NCAA_PHASE6_LR',
    'NCAA_PHASE6_INCLUDE_TOURNEY_EDGES',
    'NCAA_PHASE6_USE_GRAPH_WINPROB',
]:
    print(f'  {k}={env[k]}')

result = subprocess.run(cmd, cwd=str(root), env=env, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Pipeline failed with code {result.returncode}')

In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
if not (root / 'src' / 'train_baseline.py').exists() and (root.parent / 'src' / 'train_baseline.py').exists():
    root = root.parent

expected = [
    root / 'features' / 'team_ratings_m.parquet',
    root / 'features' / 'team_ratings_w.parquet',
    root / 'features' / 'graph_embeddings_m.parquet',
    root / 'features' / 'graph_embeddings_w.parquet',
    root / 'features' / 'graph_features_m.csv',
    root / 'features' / 'graph_features_w.csv',
    root / 'oof' / 'oof_logreg_m.csv',
    root / 'oof' / 'oof_logreg_w.csv',
    root / 'oof' / 'oof_stack_m.csv',
    root / 'oof' / 'oof_stack_w.csv',
    root / 'oof' / 'oof_stack_tourney_m.csv',
    root / 'oof' / 'oof_stack_tourney_w.csv',
    root / 'oof' / 'oof_stack_cal_best_m.csv',
    root / 'oof' / 'oof_stack_cal_best_w.csv',
    root / 'eval' / 'fold_metrics_m.csv',
    root / 'eval' / 'fold_metrics_w.csv',
    root / 'submissions' / 'submission_template.csv',
]

for p in expected:
    print(f'{p.name:36s} ->', p.exists())

for gender in ['m', 'w']:
    metrics_path = root / 'eval' / f'fold_metrics_{gender}.csv'
    if not metrics_path.exists():
        continue
    df = pd.read_csv(metrics_path)
    top = df.groupby(['model', 'regime'], as_index=False)[['brier', 'log_loss']].mean().sort_values('brier').head(12)
    print(f'\nTop metrics ({gender.upper()}):')
    display(top)

    cal_only = top[top['model'].str.contains('_cal_')]
    if not cal_only.empty:
        print(f'Best calibrated rows ({gender.upper()}):')
        display(cal_only.head(6))